In [ ]:
# Colab bootstrap — auto-clone repo on Google Colab, no-op locally
import os, sys, subprocess

REPO = "https://github.com/jongmoonha/AI-PHM_Graduate.git"
DIR  = "AI-PHM_Graduate"

try:
    import google.colab  # type: ignore
    target = '/content/' + DIR
    if not os.path.isdir(target):
        subprocess.run(["git", "clone", REPO, target], check=True)
    os.chdir(target)
    print('Google Colab detected. Working directory:', os.getcwd())
except ImportError:
    print('Local environment detected. Working directory:', os.getcwd())


# 베어링 결함 진단: 커토그램, 대역필터와 포락선 분석, MED

베어링 결함 신호는 공진 대역에서 주기적인 충격에 의해 변조된 형태로 나타난다. 커토그램으로 최적의 대역을 찾아 대역필터를 적용한 뒤, 힐베르트 포락선 스펙트럼에서 결함 주파수를 확인한다. 마지막으로 최소 엔트로피 역컨볼루션(minimum entropy deconvolution, MED)을 적용하여 충격 성분을 강화한다.

## 학습 목표
- 베어링 결함 특성 주파수(BPFO, BPFI)를 계산한다.
- 커토그램으로 결함 신호의 공진 대역을 탐지한다.
- 대역필터와 힐베르트 포락선 스펙트럼으로 결함 주파수를 확인한다.
- MED 를 적용하기 전과 후의 충격 강화 효과를 비교한다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import kurtogram
from utils import fft, filtering, filtering_zerophase, hilbert_envelope, MEDA

---

## 실습 1. 베어링 특성 주파수 계산

CWRU Drive End 베어링의 배수는 다음과 같다.
- BPFI = 5.4152 × f_shaft (내륜)
- BPFO = 3.5848 × f_shaft (외륜)

In [ ]:
rpm = 1772
f_shaft = rpm / 60
DE_BPFI = 5.4152 * f_shaft
DE_BPFO = 3.5848 * f_shaft
print(f'Shaft Frequency : {f_shaft:.2f} Hz')
print(f'DE BPFI (Inner) : {DE_BPFI:.2f} Hz')
print(f'DE BPFO (Outer) : {DE_BPFO:.2f} Hz')

---

## 실습 2. 데이터 로드

`data_fault_DE_IR.csv` 는 Drive End 내륜(IR) 결함 신호(fs = 12000 Hz)이다.

In [ ]:
fs = 12000
data = np.array(pd.read_csv('./data/data_fault_DE_IR.csv'))
y = data[:, 1]
N = len(y)
t = np.arange(N) / fs
print(f'Number of samples N = {N}, duration = {N/fs:.3f} s')

# --- 원 신호 시각화 ---
f_y, A_y = fft(y, fs)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 6))
ax1.plot(t, y, color='C0')
ax1.set_xlabel('Time (s)'); ax1.set_ylabel('y')
ax1.set_title('Raw signal (IR fault)')
ax2.plot(f_y, A_y, color='C0')
ax2.set_xlabel('Frequency (Hz)'); ax2.set_ylabel('|Y|')
ax2.set_title('Spectrum')
fig.tight_layout()
plt.show()

원 신호의 스펙트럼에서는 결함 주파수(약 160 Hz)를 직접 확인하기 어렵다. 결함에 의한 충격이 공진 대역의 진동을 진폭 변조하는 형태로 나타나기 때문이다.

---

## 실습 3. 커토그램으로 공진 대역 탐지

스펙트럼 첨도(spectral kurtosis)를 여러 분해 레벨(level)에서 계산하여 충격성이 가장 강한 주파수 대역을 찾는다.

`nlevel=7` 은 커토그램의 이진 트리 분해 깊이를 뜻하며, 이때 주파수 축은 최대 $2^7 = 128$개의 대역으로 분해된다. 데이터 길이 $N$ 은 조건 $\log_2 N - 7 > 0$ 을 만족해야 한다.

In [ ]:
nlevel = 7
Kwav, Level_w, freq_w, c, max_Kurt, bandwidth, level_max = kurtogram.fast_kurtogram(y.copy(), fs, nlevel=nlevel)

minw = np.where(Level_w == level_max)[0][0]
kurtw = np.where(Kwav[minw, :] == max_Kurt)[0][0]
bandw = freq_w[kurtw]
f_center = bandw + bandwidth / 2

fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(Kwav, interpolation='none', aspect='auto')
xticks = np.linspace(0, fs/2, nlevel + 1).astype(int)
ax.set_xticks(np.linspace(0, Kwav.shape[1] - 1, nlevel + 1))
ax.set_xticklabels(xticks)
ax.set_yticks(np.arange(0, Kwav.shape[0]))
ax.set_yticklabels(np.round(Level_w, 1))
ax.set_xlabel('Frequency (Hz)'); ax.set_ylabel('Level')
ax.set_title(f'K max = {max_Kurt:.3f} at level {level_max:.1f}\n'
             f'Center = {f_center:.1f} Hz, BW = {bandwidth:.1f} Hz')
fig.colorbar(im, ax=ax, label='Spectral Kurtosis')
fig.tight_layout()
plt.show()

---

## 실습 4. 탐지한 대역으로 대역필터링

커토그램이 찾은 [bandw, bandw+bandwidth] 대역을 Butterworth 대역필터로 추출한다.

In [ ]:
# Kurtogram 이 선택한 대역을 bandpass 로 사용
# Nyquist 경계를 넘으면 fs/2 바로 아래로 clamp
b1 = max(bandw, 1.0)
b2 = bandw + bandwidth
if b2 > fs / 2:
    b2 = fs / 2 - 1
f_low, f_high = b1, b2
print(f'Bandpass: [{f_low:.1f}, {f_high:.1f}] Hz')

y_bp = filtering_zerophase(y, fs, 'band', f_low=f_low, f_high=f_high, order=4)
f_bp, A_bp = fft(y_bp, fs)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 6))
ax1.plot(t, y_bp, color='C1')
ax1.set_xlabel('Time (s)'); ax1.set_ylabel('Amplitude')
ax1.set_title(f'Bandpass [{f_low:.0f}, {f_high:.0f}] Hz — time domain')

ax2.plot(f_bp, A_bp, color='C1')
ax2.set_xlim(0, fs/2); ax2.set_xlabel('Frequency (Hz)'); ax2.set_ylabel('Amplitude')
ax2.set_title('Bandpass FFT')
plt.tight_layout(); plt.show()

---

## 실습 5. 힐베르트 포락선 스펙트럼

대역필터를 적용한 신호에서 포락선, 곧 해석 신호의 크기를 구하고 FFT 를 계산하면 변조 주파수인 결함 주파수의 성분이 나타난다.

In [ ]:
env_bp = hilbert_envelope(y_bp)
f_env, A_env = fft(env_bp - np.mean(env_bp), fs)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 6))
ax1.plot(t, y_bp, color='C1', label='bandpass')
ax1.plot(t, env_bp, color='C3', label='envelope')
ax1.set_xlabel('Time (s)'); ax1.set_ylabel('amp')
ax1.legend()

ax2.plot(f_env, A_env, color='C3')
ax2.set_xlim(0, 500)
for n in range(1, 4):
    ax2.axvline(n * DE_BPFI, color='C2', linestyle='--', alpha=0.7,
                label='BPFI harmonics' if n == 1 else None)
ax2.set_xlabel('Frequency (Hz)'); ax2.set_ylabel('|Env|')
ax2.set_title('Envelope Spectrum (dashed: BPFI Harmonics)')
ax2.legend()
fig.tight_layout()
plt.show()

포락선 스펙트럼에서 BPFI(약 160 Hz)와 그 고조파(320, 480 Hz)에 성분이 나타나면 내륜 결함으로 진단할 수 있다. 같은 그래프에서 가장 큰 성분은 축 회전 주파수(약 29.6 Hz)의 성분이고, BPFI 성분은 그 약 0.6배이다. 이것은 내륜이 축과 함께 회전하여 결함이 한 회전마다 하중 구역을 통과하고, 그 결과 충격의 크기가 축 회전 주기로 변조되기 때문이다. 축 회전 주파수 성분이 BPFI 성분보다 크다고 해서 대역을 잘못 고른 것은 아니다.

---

## 실습 6. MED 와 두 방법의 비교

MED 는 출력의 충격성이 가장 강해지는 필터를 데이터로부터 구한다.
대역을 선택하지 않고 원 신호에서 바로 충격을 강화하는 대안 기법이다.

In [ ]:
FilterSize = 50
_, y_MED = MEDA(y, FilterSize=FilterSize, remain=True)
y_MED[:FilterSize] = 0

f_MED, A_MED = fft(y_MED, fs)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 6))
ax1.plot(t, y_MED, color='C4')
ax1.set_xlabel('Time (s)'); ax1.set_ylabel('y_MED')
ax1.set_title(f'MED filtered (FilterSize={FilterSize})')
ax2.plot(f_MED, A_MED, color='C4')
ax2.set_xlabel('Frequency (Hz)'); ax2.set_ylabel('|Y|')
ax2.set_title('Spectrum of MED output')
fig.tight_layout()
plt.show()

**관찰 (MED)**: MED 출력 `y_MED` 는 원 신호를 충격 성분이 강화되도록 최적화한 FIR 필터에 통과시킨 결과이다. 시간 파형의 **충격성**은 대역필터 출력보다 강하다. 이어지는 포락선 스펙트럼에서는 BPFI(약 160 Hz) 성분이 축 회전 주파수(약 29.6 Hz) 성분보다 조금 커서 가장 큰 성분이 되며, 축 회전 주파수 성분이 BPFI 성분보다 컸던 대역필터 경로(실습 5)와 이 점이 다르다. 마지막 셀이 출력하는 BPFI peak/floor 도 MED 경로가 약 두 배이다.

**주**: FIR 필터의 길이(`FilterSize`)는 신호의 특성에 따라 조정해야 한다. 이 데이터에서는 필터 길이가 충격의 주기(약 75 샘플)보다 짧을 때 잘 동작하며, `FilterSize` 를 25, 50으로 하면 BPFI peak/floor 가 각각 약 56, 52이다. 필터 길이가 주기와 같은 75이면 BPFI 성분이 축 회전 주파수 성분보다 작아지고, 100 ~ 400에서는 BPFI 변조가 거의 사라진다(peak/floor 약 2.2 ~ 4.6). 500 이상으로 매우 길게 하면 이 비가 다시 10 안팎으로 커지지만, 25, 50일 때의 5분의 1 수준이다.

In [ ]:
env_MED = hilbert_envelope(y_MED)
f_menv, A_menv = fft(env_MED - np.mean(env_MED), fs)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 6))
ax1.plot(t, y_MED, color='C4', label='MED')
ax1.plot(t, env_MED, color='C3', label='envelope')
ax1.set_xlabel('Time (s)'); ax1.set_ylabel('amp')
ax1.legend()

ax2.plot(f_menv, A_menv, color='C3')
ax2.set_xlim(0, 500)
for n in range(1, 4):
    ax2.axvline(n * DE_BPFI, color='C2', linestyle='--', alpha=0.7,
                label='BPFI harmonics' if n == 1 else None)
ax2.set_xlabel('Frequency (Hz)'); ax2.set_ylabel('|Env|')
ax2.set_title('Envelope spectrum of MED output')
ax2.legend()
fig.tight_layout()
plt.show()

In [ ]:
# 5단계 비교: raw / bandpass / BP envelope / MED / MED envelope (5-stage comparison)
fig, axes = plt.subplots(5, 1, figsize=(13, 10), sharex=True)
panels = [
    ('Raw',          y,         'C0'),
    ('SK+BP',        y_bp,     'C1'),
    ('SK+BP env',    env_bp,   'C3'),
    ('MED',          y_MED,    'C4'),
    ('MED envelope', env_MED,  'C2'),
]
for ax, (label, sig, color) in zip(axes, panels):
    t_plot = np.arange(len(sig)) / fs
    ax.plot(t_plot, sig, color=color, lw=0.6)
    ax.set_ylabel('Amplitude'); ax.set_title(label)
axes[-1].set_xlabel('Time (s)')
axes[0].set_xlim(0, 0.2)
fig.suptitle('5-Stage Time-Domain Comparison')
plt.tight_layout(); plt.show()

### 두 경로의 포락선 스펙트럼 비교

커토그램이 고른 대역의 대역필터 출력에서 포락선을 구하는 경로와 MED 출력에서 포락선을 구하는 경로의 포락선 스펙트럼을 나란히 놓고, BPFI 의 고조파 위치에 성분이 나타나는지 확인한다.

In [ ]:
# 두 경로의 포락선 스펙트럼 비교
fig, ax = plt.subplots(1, 2, figsize=(14, 4), sharey=False)
f_cmp_bp, A_cmp_bp = fft(env_bp - np.mean(env_bp), fs)
f_cmp_med, A_cmp_med = fft(env_MED - np.mean(env_MED), fs)

ax[0].plot(f_cmp_bp, A_cmp_bp, 'C1')
ax[0].set_title('Spectral Kurtosis + Band-pass + Envelope')
ax[0].set_xlim(0, 500)

ax[1].plot(f_cmp_med, A_cmp_med, 'C4')
ax[1].set_title('MED + Envelope')
ax[1].set_xlim(0, 500)

for a in ax:
    for k in range(1, 6):
        a.axvline(k * DE_BPFI, color='C2', ls='--', alpha=0.5)
    a.set_xlabel('Frequency (Hz)')
    a.set_ylabel('Amplitude')

fig.tight_layout()
plt.show()

# 정량 비교: BPFI 피크 대 배경 ratio
def peak_ratio(f, A, f0, df=2.0):
    """피크(=f0±df) / 넓은 주변 floor(=f0±5df 를 제외한 f0-50 ~ f0+50)."""
    m_peak = (f > f0 - df) & (f < f0 + df)
    peak = A[m_peak].max() if m_peak.any() else 0.0
    m_floor = ((f > f0 - 50) & (f < f0 + 50)
               & ~((f > f0 - 5*df) & (f < f0 + 5*df)))
    floor = np.median(A[m_floor]) if m_floor.any() else 1e-12
    return peak / max(floor, 1e-12)

print(f'BPFI peak/floor — BP envelope  : {peak_ratio(f_cmp_bp,  A_cmp_bp,  DE_BPFI):.2f}')
print(f'BPFI peak/floor — MED envelope : {peak_ratio(f_cmp_med, A_cmp_med, DE_BPFI):.2f}')

## 정리

- **커토그램**: 결함 충격이 집중된 최적의 공진 대역을 자동으로 탐지한다.
- **대역필터와 힐베르트 포락선**: 탐지한 대역에서 변조 주파수, 곧 결함 주파수의 성분을 찾는다.
- **MED**: 데이터로부터 구한 필터로 충격 성분을 직접 강화한다.
- 두 경로 모두 포락선 스펙트럼에서 BPFI 와 그 고조파의 성분이 나타나는 것을 확인한다.

### 생각해보기
1. 커토그램의 `nlevel` 을 바꾸면 찾은 중심 주파수와 대역폭은 어떻게 달라지는가?
2. MED 의 `FilterSize` 를 10, 100, 500으로 바꾸면 출력에서 충격의 선명도는 어떻게 변하는가?
3. 외륜(OR) 결함 신호(`data_fault_FE_OR.csv`)는 Fan End 베어링에서 측정한 것이며, 이 베어링에서는 BPFO = 3.0530 × f_shaft 이다. 이 신호에 같은 파이프라인을 적용하면 어떤 주파수의 성분이 나타나야 하는가?